# Code-Mixed Pedagogical Flow Extractor
## Subtask 2: Audio Extraction & Transcription

**Instructions for Kaggle:**
1. Under the **Settings** panel on the right, change the **Accelerator** to `GPU T4 x2`.
2. On the right panel, click **Add Data**, select **Upload**, and upload your `metadata.json` file. Name the dataset something like `irel-metadata`.
3. Change the `METADATA_PATH` variable in the second python cell to point to your uploaded `metadata.json` file (e.g. `/kaggle/input/irel-metadata/metadata.json`).
4. Click **Run All**.
5. Once finished, download the generated JSON files from `/kaggle/working/data/interim/`.

In [ ]:
!pip install -U yt-dlp openai-whisper ffmpeg-python

In [ ]:
import json
import os
import subprocess
import torch
import whisper

# => IMPORTANT: Update this path to match where your dataset is uploaded! <=
METADATA_PATH = '/kaggle/input/datasets/akshatatkaggle/irel-task-metadata-subtask1/metadata.json'
OUTPUT_DIR = '/kaggle/working/data/interim'
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch is using device: {DEVICE.upper()}")
if DEVICE == 'cpu':
    print("WARNING: GPU is not detected! Make sure you selected 'GPU T4 x2' as your Accelerator in Kaggle settings.")

print("Loading whisper-large-v3 model. This may take a minute...")
model = whisper.load_model('large-v3', device=DEVICE)

for video in metadata:
    video_id = video['video_id']
    url = video['video_url']
    print(f'\n========================================')
    print(f'Processing {video_id}')
    print(f'URL: {url}')
    
    audio_file = f'/kaggle/working/{video_id}.webm'
    wav_file = f'/kaggle/working/{video_id}.wav'
    
    # 1. Download audio using yt-dlp
    print('-> Downloading audio stream...')
    subprocess.run(['yt-dlp', '-f', 'bestaudio', '-o', audio_file, url])
    
    if not os.path.exists(audio_file):
        # Fallback to m4a if webm is not available
        audio_file = f'/kaggle/working/{video_id}.m4a'
        subprocess.run(['yt-dlp', '-f', 'm4a', '-o', audio_file, url])
        
    # 2. Convert to 16kHz mono wav via ffmpeg
    print('-> Downsampling to 16kHz WAV...')
    subprocess.run(['ffmpeg', '-y', '-i', audio_file, '-ar', '16000', '-ac', '1', wav_file])
    
    if not os.path.exists(wav_file):
        print(f'Error: Failed to process audio for {video_id}. Skipping.')
        continue
        
    # 3. Transcribe via Whisper (FP16 is the standard default for Whisper on GPU)
    print('-> Transcribing with Whisper (language="hi", task="transcribe")...')
    result = model.transcribe(wav_file, language='hi', task='transcribe', word_timestamps=True)
    
    # 4. Save results
    out_path = os.path.join(OUTPUT_DIR, f'v1_raw_transcript_{video_id}.json')
    with open(out_path, 'w', encoding='utf-8') as f:
        # Save full dictionary: result['text'] and result['segments']
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f'-> Saved transcript to {out_path}')
    
    # Clean up to save working directory space
    if os.path.exists(audio_file): os.remove(audio_file)
    if os.path.exists(wav_file): os.remove(wav_file)

print('\n========================================')
print('All Subtask 2 transcriptions completed!')
print('Please download the output JSON files from the /kaggle/working/data/interim folder.')
